In [143]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [114]:
train = pd.read_csv("../data/processed/train_clean.csv")
test = pd.read_csv("../data/processed/test_clean.csv")

In [ ]:
categorical_cols = ["Contract Length", "Subscription Type", "Gender"]

numeric_cols = [
    "Total Spend",
    "Support Calls",
    "Usage Frequency",
    "Age",
    "Last Interaction",
    "Tenure",
    "Payment Delay",
]

rf_features = categorical_cols + numeric_cols

In [122]:
rf_preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    # (StandardScaler(), numeric_cols),
    # remainder="drop"
    remainder="passthrough"
)

In [123]:
rf_model = RandomForestClassifier(  
    max_depth=15, 
    min_samples_leaf=5, 
    random_state=1234
)

In [124]:
rf_pipeline = make_pipeline(
    rf_preprocessor,
    rf_model
)

In [ ]:
X = train[rf_features].copy()
y = train["Churn"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1234,
    stratify=y
)

In [126]:
rf_fit = rf_pipeline.fit(X_train, y_train)

val_pred_rf = rf_fit.predict_proba(X_val)[:, 1]
auc_rf = roc_auc_score(y_val, val_pred_rf)
auc_rf
#0.9473820204891165
#0.9484317692630228

0.9484317692630228

In [135]:
X_test = test[rf_features].copy()
y_val_pred = rf_fit.predict(X_val)
test_pred_rf = rf_fit.predict_proba(X_test)[:, 1]

In [103]:
submission_rf = pd.DataFrame({
    "CustomerID": test["CustomerID"],
    "Churn": test_pred_rf
})

submission_rf.to_csv("../data/submissions/random_forest.csv", index=False)

In [132]:
y_pred = rf_fit.predict(X_val)


In [137]:
accuracy = accuracy_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
conf_matrix = confusion_matrix(y_val, y_val_pred)

#print
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"ROC AUC: {auc_rf:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 0.8968
F1-score: 0.8484
ROC AUC: 0.9484
Confusion Matrix:
[[36857  5581]
 [  675 17512]]
